## Imports

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from create_event_sequence import create_event_sequence
from flood_adapt import FloodAdapt
from flood_adapt.config.config import Settings

## Config

In [2]:
# FloodAdapt databse inputs
DATA_DIR = Path(r"c:\Users\athanasi\Github\Database\Working_Databases\Charleston\4_FloodAdapt\Database")
site = "charleston_beta_release"

# Monte Carlo input parameters for event sequences
timestep = 1  # timestep of Monte Carlo in years
sim_time = 30  # length of simulation in years
no_seq = 20  # number of event sequences to generate
seed = 42  # random seed for reproducibility

# SLR scenario to evaluate damages over time
slr_scenario_name = "NOAA High" # This needs to be a scenario in the FloodAdapt database

# lookup table
ds_impacts = xr.open_dataset("lookup_table_charleston.nc")

In [3]:
times = 2020 + np.linspace(0, sim_time, int(sim_time / timestep) + 1)
times

array([2020., 2021., 2022., 2023., 2024., 2025., 2026., 2027., 2028.,
       2029., 2030., 2031., 2032., 2033., 2034., 2035., 2036., 2037.,
       2038., 2039., 2040., 2041., 2042., 2043., 2044., 2045., 2046.,
       2047., 2048., 2049., 2050.])

## Initialize FloodAdapt Database

In [4]:
settings = Settings(
    DATABASE_ROOT=DATA_DIR,
    DATABASE_NAME=site,
)
fa = FloodAdapt(database_path=settings.database_path)
slr_years = [fa.interp_slr(slr_scenario=slr_scenario_name, year=time) for time in times]

2025-12-16 11:40:13 AM - FloodAdapt.Database - INFO - Initializing database to charleston_beta_release at c:\users\athanasi\github\database\working_databases\charleston\4_floodadapt\database


In [5]:
ds_impacts

<xarray.Dataset> Size: 427MB
Dimensions:       (object_id: 61858, slr: 4, strategy: 2, event: 4)
Coordinates:
  * object_id     (object_id) <U1598 395MB '14048' '67199' ... '70438' '70444'
  * slr           (slr) int32 16B 0 1 2 3
  * strategy      (strategy) <U16 128B 'no_measures' 'floodproof_all_0'
  * event         (event) <U10 160B 'subevent_1' 'subevent_2' ... 'subevent_4'
Data variables:
    inun_depth    (object_id, slr, strategy, event) float64 16MB ...
    total_damage  (object_id, slr, strategy, event) float64 16MB ...

In [6]:
# ABM Simulation: Household Floodproofing Decisions (integrated logic)
from abm_simulator import ABMSimulator

# Instantiate the simulator (event sequence and damage lookup are handled internally)
abm = ABMSimulator(
    ds_impacts=ds_impacts,
    times=times,
    slr_times=slr_years,
    no_seq=no_seq,
    damage_threshold=0.3,  # You can change this threshold as needed
    seed=seed
)

In [7]:
# Run the simulation
abm.run("floor ")

[TIMER] Initialized arrays in 0.000 seconds
[BASELINE] Evaluating sequence 1/20...
[TIMER] Year 1/31 in sequence 1 took 0.000 seconds
[TIMER] slr_damage_lookup for event 'subevent_3' in year 1 took 0.232 seconds
[TIMER] Year 2/31 in sequence 1 took 0.233 seconds
[TIMER] Year 3/31 in sequence 1 took 0.000 seconds
[TIMER] slr_damage_lookup for event 'subevent_2' in year 3 took 0.203 seconds
[TIMER] Year 4/31 in sequence 1 took 0.204 seconds
[TIMER] Year 5/31 in sequence 1 took 0.000 seconds
[TIMER] slr_damage_lookup for event 'subevent_2' in year 5 took 0.204 seconds
[TIMER] Year 6/31 in sequence 1 took 0.205 seconds
[TIMER] Year 7/31 in sequence 1 took 0.000 seconds
[TIMER] slr_damage_lookup for event 'subevent_2' in year 7 took 0.203 seconds
[TIMER] Year 8/31 in sequence 1 took 0.203 seconds
[TIMER] Year 9/31 in sequence 1 took 0.000 seconds
[TIMER] slr_damage_lookup for event 'subevent_2' in year 9 took 0.205 seconds
[TIMER] Year 10/31 in sequence 1 took 0.205 seconds
[TIMER] Year 11/

KeyboardInterrupt: 

In [ ]:
abm.plot_event_damage_timeseries(1)

In [ ]:
abm.plot_total_damage_statistics()